In [68]:
!dpkg -l | grep tensorrt

In [70]:
!pip install nvidia-tensorrt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.9 MB/s eta 0:00:00
  Created wheel for tensorrt: filename=tensorrt-11.1.0.106-py3-none-any.whl size=16564 sha256=1b475c386f86ad091e0afa41899973dca4e5122134e6d25f49933e51eeeb896a
  Stored in directory: /root/.cache/pip/wheels/64/60/e6/9df6cdc73f3d2d55f99b57987892a29d470494bf419be000ad
  Created wheel for tensorrt_cu13: filename=tensorrt_cu13-11.1.0.106-py3-none-any.whl size=23074 sha256=06877c4de12dc9e5d05c622498a84f2fda867b7fcefc9df16e8a8ab310fe41c6
  Stored in directory: /root/.cache/pip/wheels/77/0f/83/d9c20d0e840bfaf19d384d48255

In [71]:
import tensorrt as trt
print(trt.__version__)

11.1.0.106


In [75]:
import tensorrt as trt

ONNX_FILE = "/content/rfdetr-seg-small.onnx"
ENGINE_FILE = "/content/model.engine"

logger = trt.Logger(trt.Logger.WARNING)
builder = trt.Builder(logger)


network = builder.create_network()

parser = trt.OnnxParser(network, logger)


with open(ONNX_FILE, "rb") as f:
    if not parser.parse(f.read()):
        print(" ONNX Parsing Failed")
        for i in range(parser.num_errors):
            print(parser.get_error(i))
        raise SystemExit()
    else:
        print("ONNX Parsed Successfully")

config = builder.create_builder_config()
config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)

print("⏳ Building engine (serialized)...")


serialized_engine = builder.build_serialized_network(network, config)

if serialized_engine is None:
    raise RuntimeError("❌ Engine build failed")

# Save engine
with open(ENGINE_FILE, "wb") as f:
    f.write(serialized_engine)

print("✅ Engine saved:", ENGINE_FILE)

✅ ONNX Parsed Successfully
⏳ Building engine (serialized)...
✅ Engine saved: /content/model.engine


In [77]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 12.5 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659496 sha256=697944a0405e7f2427bf1d5bf78b8a0873742a0ca88fe0837c3b66916644f9cf
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [138]:
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit
import numpy as np
import cv2

ENGINE_PATH = "/content/model.engine"

logger = trt.Logger(trt.Logger.WARNING)
runtime = trt.Runtime(logger)

with open(ENGINE_PATH, "rb") as f:
    engine = runtime.deserialize_cuda_engine(f.read())

context = engine.create_execution_context()

print("✅ Engine loaded")

✅ Engine loaded


In [139]:
inputs = []
output_buffers = []
bindings = []

for i in range(engine.num_io_tensors):

    name = engine.get_tensor_name(i)

    shape = context.get_tensor_shape(name)
    size = trt.volume(shape)

    dtype = trt.nptype(engine.get_tensor_dtype(name))

    host_mem = cuda.pagelocked_empty(size, dtype)
    device_mem = cuda.mem_alloc(host_mem.nbytes)

    bindings.append(int(device_mem))

    if engine.get_tensor_mode(name) == trt.TensorIOMode.INPUT:

        inputs.append({
            "host": host_mem,
            "device": device_mem
        })

    else:

        output_buffers.append({
            "host": host_mem,
            "device": device_mem
        })

In [140]:
def preprocess(image_path, input_shape=(384, 384)):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, input_shape)

    img = img.astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))  # CHW format
    img = np.expand_dims(img, axis=0)    # NCHW

    return img

In [141]:
def run_inference(image_path):

    input_data = preprocess(image_path)

    np.copyto(inputs[0]["host"], input_data.ravel())

    cuda.memcpy_htod(
        inputs[0]["device"],
        inputs[0]["host"]
    )

    context.execute_v2(bindings)

    results = []

    output_index = 0

    for i in range(engine.num_io_tensors):

        name = engine.get_tensor_name(i)

        if engine.get_tensor_mode(name) != trt.TensorIOMode.OUTPUT:
            continue

        cuda.memcpy_dtoh(
            output_buffers[output_index]["host"],
            output_buffers[output_index]["device"]
        )

        shape = tuple(context.get_tensor_shape(name))

        tensor = output_buffers[output_index]["host"].reshape(shape)

        results.append(tensor)

        output_index += 1

    return results

In [142]:
print("="*60)

print("Number of IO Tensors:", engine.num_io_tensors)

print("="*60)

for i in range(engine.num_io_tensors):

    name = engine.get_tensor_name(i)

    print(f"Tensor {i}")
    print("Name :", name)
    print("Mode :", engine.get_tensor_mode(name))
    print("Shape:", engine.get_tensor_shape(name))
    print("Dtype:", engine.get_tensor_dtype(name))
    print("-"*40)

Number of IO Tensors: 4
Tensor 0
Name : input
Mode : TensorIOMode.INPUT
Shape: (1, 3, 384, 384)
Dtype: DataType.FLOAT
----------------------------------------
Tensor 1
Name : dets
Mode : TensorIOMode.OUTPUT
Shape: (1, 100, 4)
Dtype: DataType.FLOAT
----------------------------------------
Tensor 2
Name : labels
Mode : TensorIOMode.OUTPUT
Shape: (1, 100, 11)
Dtype: DataType.FLOAT
----------------------------------------
Tensor 3
Name : masks
Mode : TensorIOMode.OUTPUT
Shape: (1, 100, 96, 96)
Dtype: DataType.FLOAT
----------------------------------------


In [143]:
IMAGE_PATH = "/content/09Q1_bmp.rf.U5uNHOrjwv3cV632bY5S.bmp"

outputs = run_inference(IMAGE_PATH)

print("✅ Inference Done")
print("Output tensors:")

outputs = run_inference(IMAGE_PATH)

# print(outputs[0].shape)
# print(outputs[1].shape)
# print(outputs[2].shape)
# for i, out in enumerate(outputs):
#     print(f"\n--- Output {i} ---")
#     print(out)  # first 50 values

✅ Inference Done
Output tensors:


In [144]:
import time
import numpy as np

times = []

for _ in range(100):

    start = time.perf_counter()

    run_inference("/content/09Q1_bmp.rf.U5uNHOrjwv3cV632bY5S.bmp")

    end = time.perf_counter()

    times.append(end-start)

avg = np.mean(times)

print("="*40)
print("TensorRT Benchmark")
print("="*40)

print(f"Latency : {avg*1000:.2f} ms")
print(f"FPS      : {1/avg:.2f}")

TensorRT Benchmark
Latency : 27.93 ms
FPS      : 35.80
